# A01: Data collection — rugby-born residents by area (Victoria, 2021)
This notebook finds which SA2 areas in Victoria have the most people born in rugby-playing countries.

In [21]:
import pandas as pd
import os
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files)

/kaggle/input []
/kaggle/input/datasets []
/kaggle/input/datasets/somahattori []
/kaggle/input/datasets/somahattori/victoria-census-2021-country-of-birth-g09-sa2 ['2021Census_G09F_VIC_SA2.csv', '2021Census_G09G_VIC_SA2.csv', '2021Census_G09H_VIC_SA2.csv']
/kaggle/input/datasets/somahattori/abs-2021-geography-names ['2021Census_geog_desc_1st_2nd_3rd_release.xlsx']


## 1. Load the data
Read the three Census files (G09F, G09G, G09H). Each has 524 rows, one per SA2 area in Victoria.

In [22]:
path = "/kaggle/input/datasets/somahattori/victoria-census-2021-country-of-birth-g09-sa2/"

f = pd.read_csv(path + "2021Census_G09F_VIC_SA2.csv")
g = pd.read_csv(path + "2021Census_G09G_VIC_SA2.csv")
h = pd.read_csv(path + "2021Census_G09H_VIC_SA2.csv")

print(f.shape, g.shape, h.shape)

(524, 201) (524, 201) (524, 191)


## 2. Keep only the columns we need
From each file, keep the area code (`SA2_CODE_2021`) and the "Persons total" column for each country, plus the total population (`P_Tot_Tot`).

In [23]:
f_filtered = f[["SA2_CODE_2021","P_Fiji_Tot", "P_England_Tot"]]
g_filtered = g[["SA2_CODE_2021", "P_New_Zealand_Tot", "P_Ireland_Tot"]]
h_filtered = h[["SA2_CODE_2021", "P_Samoa_Tot", "P_Scotland_Tot", "P_Wales_Tot", 
                "P_South_Africa_Tot", "P_Tot_Tot"]]
print(f_filtered.shape, g_filtered.shape, h_filtered.shape)
h_filtered.head()

(524, 3) (524, 3) (524, 6)


,SA2_CODE_2021,P_Samoa_Tot,P_Scotland_Tot,P_Wales_Tot,P_South_Africa_Tot,P_Tot_Tot
0,201011001,0,47,12,74,16835
1,201011002,3,47,13,26,12131
2,201011005,0,32,5,14,7261
3,201011006,3,25,6,8,10661
4,201011007,0,20,0,6,4230


## 3. Merge the three files
Join the three tables on SA2_CODE_2021, so each area has all eight countries in one row. The result has 10 columns; the area code, eight countries and the total population.

In [24]:
merged_tables = f_filtered.merge(g_filtered, on="SA2_CODE_2021").merge(h_filtered, on="SA2_CODE_2021")
merged_tables.shape
print(merged_tables)

     SA2_CODE_2021  P_Fiji_Tot  P_England_Tot  P_New_Zealand_Tot  \
0        201011001          20            333                151   
1        201011002           3            282                130   
2        201011005           5            268                 64   
3        201011006           4            161                100   
4        201011007           0            102                 30   
..             ...         ...            ...                ...   
519      217041478           3            238                157   
520      217041479          12            358                233   
521      217041480           8            237                148   
522      297979799           0              0                  0   
523      299999499           4            170                157   

     P_Ireland_Tot  P_Samoa_Tot  P_Scotland_Tot  P_Wales_Tot  \
0               25            0              47           12   
1               38            3              47        

## 4. Count people born in rugby-playing countries
Add the eight country columns into one column, `rugby_born`.
Check: the total for Victoria should be 373,250.

In [25]:
countries = ["P_Fiji_Tot", "P_England_Tot", "P_New_Zealand_Tot",
"P_Ireland_Tot", "P_Samoa_Tot", "P_Scotland_Tot", "P_Wales_Tot", "P_South_Africa_Tot"]
merged_tables["rugby_born"] = merged_tables[countries].sum(axis=1)
print(merged_tables["rugby_born"].sum())

373250


In [26]:
# Check against ABS QuickStats for Victoria (2021, Country of birth)
quickstats = {"P_England_Tot": 174552, "P_New_Zealand_Tot": 99344}

for col, official in quickstats.items():
    ours = merged_tables[col].sum()
    diff = ours - official
    print(col, "ours:", ours, "QuickStats:", official,
          "diff:", diff, f"({diff / official * 100:.2f}%)")

P_England_Tot ours: 174596 QuickStats: 174552 diff: 44 (0.03%)
P_New_Zealand_Tot ours: 99293 QuickStats: 99344 diff: -51 (-0.05%)


## 5. Remove areas with very few residents
Keep only areas with at least 1,000 people. In Victoria, 14 SA2 areas have fewer than 250 residents, and the next smallest area has about 1,900, so 1,000 is a clear cut-off.


In [27]:
print((merged_tables["P_Tot_Tot"] < 500).sum())
print((merged_tables["P_Tot_Tot"] < 1000).sum())

14
14


Both checks return 14, so no SA2 area in Victoria has between 500 and 1,000 people. Whether the cut-off is 500 or 1,000, the same 14 areas are removed.

In [28]:
areas = merged_tables[merged_tables["P_Tot_Tot"] >= 1000].copy()
print(areas.head())

   SA2_CODE_2021  P_Fiji_Tot  P_England_Tot  P_New_Zealand_Tot  P_Ireland_Tot  \
0      201011001          20            333                151             25   
1      201011002           3            282                130             38   
2      201011005           5            268                 64             11   
3      201011006           4            161                100              7   
4      201011007           0            102                 30              8   

   P_Samoa_Tot  P_Scotland_Tot  P_Wales_Tot  P_South_Africa_Tot  P_Tot_Tot  \
0            0              47           12                  74      16835   
1            3              47           13                  26      12131   
2            0              32            5                  14       7261   
3            3              25            6                   8      10661   
4            0              20            0                   6       4230   

   rugby_born  
0         662  
1         54

## 6. Calculate the share
`rugby_share` = `rugby_born` / `P_Tot_Tot` × 100 (%).
The share lets us compare areas of different sizes. A large area can have many rugby-born residents simply because more people live there.

For comparison, `vic_share` is the share for Victoria as a whole.

In [29]:
areas["rugby_share"] = (areas["rugby_born"] / areas["P_Tot_Tot"] * 100).round(2)
print(areas["rugby_share"].head())

0    3.93
1    4.47
2    5.50
3    2.95
4    3.92
Name: rugby_share, dtype: float64


In [30]:
vic_share = merged_tables["rugby_born"].sum() / merged_tables["P_Tot_Tot"].sum() * 100
print(round(vic_share, 2))

5.74


## 7. Add area names
Join the SA2 names from the ABS geography file, so the results are readable.
The geography file lists every type of area in Australia (states, SA2, SA3 and so on). I keep only the SA2 rows and two columns: the area code and the area name.

The codes are changed to numbers, so they match the codes in `areas`. Then the names are joined with a left merge, which keeps all 510 areas and adds one column.

Check:
- 2,472 SA2 areas in Australia
- `areas` is now (510, 13): same rows, one new column
- 0 areas without a name

In [32]:
geo_path = "/kaggle/input/datasets/somahattori/abs-2021-geography-names/"
geo_all = pd.read_excel(geo_path + "2021Census_geog_desc_1st_2nd_3rd_release.xlsx", sheet_name=None)
for name, df in geo_all.items():
    print(name, df.shape)
    print(df.head(3))

2021_ASGS_MAIN_Structures (64791, 5)
  ASGS_Structure Census_Code_2021 AGSS_Code_2021 Census_Name_2021  \
0            AUS              AUS            AUS        AUSTRALIA   
1            STE                1              1  New South Wales   
2            STE                2              2         Victoria   

      Area sqkm  
0  7.688095e+06  
1  8.007977e+05  
2  2.274962e+05  
2021_ASGS_GCCSA_Structure (44, 5)
  ASGS_Structure Census_Code_2021 AGSS_Code_2021 Census_Name_2021  \
0            AUS              AUS            AUS        AUSTRALIA   
1            STE                1              1  New South Wales   
2            STE                2              2         Victoria   

      Area sqkm  
0  7.688095e+06  
1  8.007977e+05  
2  2.274962e+05  
2021_ASGS_Indigenous_Structures (1636, 5)
  ASGS_Structure Census_Code_2021 AGSS_Code_2021 Census_Name_2021  \
0            AUS              AUS            AUS        AUSTRALIA   
1            STE                1              1  N

In [33]:
#1. Keep only SA2 rows (area code and name)
geo = geo_all["2021_ASGS_MAIN_Structures"]
sa2 = geo[geo["ASGS_Structure"] == "SA2"][["Census_Code_2021", "Census_Name_2021"]].copy()

#2.Rename the columns to match areas, and make the code a number
sa2.columns = ["SA2_CODE_2021", "SA2_NAME"]
sa2["SA2_CODE_2021"] = sa2["SA2_CODE_2021"].astype(int)
print(sa2.shape)

#3. Add the names to areas
areas = areas.merge(sa2, on="SA2_CODE_2021", how="left")
print(areas.shape)
print(areas["SA2_NAME"].isna().sum())

(2472, 2)
(510, 13)
0


## 8. Top 10 areas
Sort by `rugby_share` and show the top 10. 

In [34]:
cols = ["SA2_CODE_2021", "P_Tot_Tot", "rugby_born", "rugby_share", 
        "P_England_Tot", "P_New_Zealand_Tot"]
areas.sort_values("rugby_share", ascending=False)[["SA2_NAME"] + cols].head(10)

,SA2_NAME,SA2_CODE_2021,P_Tot_Tot,rugby_born,rugby_share,P_England_Tot,P_New_Zealand_Tot
458,Mornington - West,214021592,10581,1484,14.03,996,157
452,Mount Eliza,214021381,18734,2526,13.48,1733,269
140,St Kilda - Central,206051513,11857,1542,13.00,622,393
457,Mornington - East,214021591,15175,1972,13.00,1307,225
453,Mount Martha,214021382,19846,2555,12.87,1746,261
180,Beaumaris,208011168,14021,1728,12.32,1008,229
141,St Kilda - West,206051514,13858,1690,12.20,736,424
185,Sandringham - Black Rock,208011173,17358,2111,12.16,1246,309
426,Point Cook - East,213051464,19274,2273,11.79,602,897
448,Skye - Sandhurst,214011376,13303,1522,11.44,671,313


**How to read the table.** Each row is one SA2 area (a neighbourhood of about 3,000–25,000 people).
- `P_Tot_Tot`: total residents
- `rugby_born`: residents born in the eight rugby-playing countries
- `rugby_share`: `rugby_born` as a percentage of total residents
- `P_England_Tot`, `P_New_Zealand_Tot`: part of `rugby_born`, shown separately to see which countries drive the total



### What the results show
- In the top 10 areas, 11.44–14.03% of residents were born in one of the eight rugby-playing countries, about 2.0–2.5 times the Victorian average (5.74%).
- They fall into three groups:
  - Mornington Peninsula and Frankston: Mornington West, Mornington East, Mount Eliza, Mount Martha, Skye - Sandhurst
  - Bayside: Beaumaris, Sandringham - Black Rock
  - St Kilda: St Kilda - Central, St Kilda - West
- In 9 of the 10 areas, England-born residents are the largest group, so this measure partly reflects where English migrants live.
- Point Cook - East is different: New Zealand-born residents (897) outnumber England-born residents (602).

### What this means
**Takeaways**
- These areas are places where rugby may already have a base, not the main target for growing new fans.
- Even in the top area, more than 85% of residents were not born in these countries, so most potential new fans are Australian-born or born elsewhere.

**Limitations**
- Country of birth is not the same as citizenship, and children of migrants born in Australia are not counted.
- The data shows where people were born, not whether they play or watch rugby.
- Share does not show size: an area with a high share may still have few people.

### Next
- A02: Build a table of rugby union clubs from Rugby Victoria's "Find My Club" (name, suburb, address, juniors), and check which of the top 10 areas have a club nearby.
- Rugby league (NRL) clubs are not included for now, because this project focuses on rugby union. They may be added later.
- Possible later checks: rank areas by the number of rugby-born residents instead of the share, and repeat the analysis for younger age groups (e.g. 15–34).

## 9. Save the result
Save all 510 areas (with names) as a CSV for the next notebook (A02).

In [35]:
areas.to_csv("/kaggle/working/a01_rugby_born_by_sa2.csv", index=False)